In [14]:
import pandas as pd
import min_features, daily_return
import importlib
importlib.reload(min_features)
importlib.reload(daily_return)


perf_df = pd.read_csv("master_run_results.csv")
perf_df['Date'] = perf_df['test_start']
returns = [1, 2, 3, 5, 10]
df_daily = daily_return.pull_daily('QQQ', returns) 
return_cols = df_daily.columns[df_daily.columns.str.contains("Return_")].to_list()
df_returns = df_daily[['Date'] + return_cols][(df_daily['Date'] < '2025-12-20') & (df_daily['Date'] > '2025-01-01')].copy()

In [63]:
r = 10
df_returns_2 = df_returns[['Date', f'Return_{r}']].sort_values(by='Date').copy()
# df has columns: ["Date", "Return_1"] where Return_1 is 0/1 (or False/True)

s = df_returns_2[f"Return_{r}"].astype(int)

# identify streak groups (new group whenever value changes)
grp = s.ne(s.shift()).cumsum()

# streak length within each group: 1,2,3,...
streak_len = s.groupby(grp).cumcount() + 1

# positive streaks for 1s, negative for 0s
df_returns_2["streak"] = streak_len.where(s.eq(1), -streak_len)
#df_returns_1.sort_values(by='Date', ascending=False)

perf_cols = ['model', 'acc', 'Date', 'train_years', 'feature_set']
performance_2 = pd.merge(df_returns_2, perf_df[(perf_df['horizon'] == r) & (perf_df['test_days'] == 1)], on='Date', how='inner')
performance_2

,Date,Return_10,streak,Unnamed: 0,run,model,test_days,bal_acc,acc,sign_acc,...,train_start,train_end,test_start,test_end,month,train_years,horizon_days,n_features,feature_set,horizon
0,2025-01-23,0,-4,2278,228,xgboost,1,1.0,1.0,1.0,...,2021-02-18,2025-01-22,2025-01-23,2025-01-23,January,4,10,80,daily,10
1,2025-01-23,0,-4,2279,228,random_forest,1,1.0,1.0,1.0,...,2021-02-18,2025-01-22,2025-01-23,2025-01-23,January,4,10,80,daily,10
2,2025-01-23,0,-4,2734,228,xgboost,1,1.0,1.0,1.0,...,2019-03-04,2025-01-22,2025-01-23,2025-01-23,January,6,10,80,daily,10
3,2025-01-23,0,-4,2735,228,random_forest,1,1.0,1.0,1.0,...,2019-03-04,2025-01-22,2025-01-23,2025-01-23,January,6,10,80,daily,10
4,2025-01-23,0,-4,5014,228,xgboost,1,1.0,1.0,1.0,...,2021-02-18,2025-01-22,2025-01-23,2025-01-23,January,4,10,87,minute,10
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
2731,2025-12-19,1,6,5017,1,random_forest,1,1.0,1.0,1.0,...,2020-01-30,2025-12-18,2025-12-19,2025-12-19,December,6,10,87,minute,10
2732,2025-12-19,1,6,7296,1,xgboost,1,1.0,1.0,1.0,...,2022-01-12,2025-12-18,2025-12-19,2025-12-19,December,4,10,167,daily+minute,10
2733,2025-12-19,1,6,7297,1,random_forest,1,1.0,1.0,1.0,...,2022-01-12,2025-12-18,2025-12-19,2025-12-19,December,4,10,167,daily+minute,10
2734,2025-12-19,1,6,7752,1,xgboost,1,0.0,0.0,-1.0,...,2020-01-30,2025-12-18,2025-12-19,2025-12-19,December,6,10,167,daily+minute,10


In [65]:
df = performance_2.copy()
# 1) Accuracy by (model, train_years, streak)
acc_piv = df.pivot_table(
    index=["model", "train_years", 'feature_set'],
    columns="streak",
    values="acc",
    aggfunc="mean"
)

# 2) Count by (model, train_years, streak)
cnt_piv = df.pivot_table(
    index=["model", "train_years", 'feature_set'],
    columns="streak",
    values="acc",          # any column works since we're counting rows
    aggfunc="size"
)

# 3) Combine into one wide table with clear column labels
out = pd.concat({"acc": acc_piv, "count": cnt_piv}, axis=1)

# optional: sort columns so streaks go -N ... -1, 1 ... N
out = out.reindex(sorted(out.columns, key=lambda x: (x[1] >= 0, x[1])), axis=1)

gcols = ["model", "train_years", "feature_set"]

df2 = df.copy()
df2["side"] = df2["streak"].gt(0).map({True: "pos", False: "neg"})  # 0 shouldn't exist with your streak logic

side_perf = (
    df2.groupby(gcols + ["side"])
       .agg(n=("acc", "size"), acc=("acc", "mean"))
       .reset_index()
)

# wide format (pos/neg columns)
side_wide = side_perf.pivot(index=gcols, columns="side", values=["acc", "n"])
side_wide

acc               n       
side                                       neg       pos   neg    pos
model         train_years feature_set                                
random_forest 4           daily         0.8000  0.912162  80.0  148.0
                          daily+minute  0.7500  0.885135  80.0  148.0
                          minute        0.1125  0.925676  80.0  148.0
              6           daily         0.7750  0.912162  80.0  148.0
                          daily+minute  0.7125  0.918919  80.0  148.0
                          minute        0.0750  0.966216  80.0  148.0
xgboost       4           daily         0.7375  0.925676  80.0  148.0
                          daily+minute  0.6625  0.891892  80.0  148.0
                          minute        0.2250  0.797297  80.0  148.0
              6           daily         0.7750  0.905405  80.0  148.0
                          daily+minute  0.6250  0.905405  80.0  148.0
                          minute        0.2375  0.804054  80.0  148.0

In [66]:
K = 3
gcols = ["model", "train_years", "feature_set"]

d = df.copy()

# keep exact -3..+3; collapse only beyond into +/-4 (representing 3+)
d["streak_bucket"] = d["streak"].clip(lower=-K, upper=K)
d.loc[d["streak"] < -K, "streak_bucket"] = -(K + 1)   # strictly less than -3
d.loc[d["streak"] >  K, "streak_bucket"] =  (K + 1)   # strictly greater than +3

flip_perf = (
    d.groupby(gcols + ["streak_bucket"])
     .agg(n=("acc", "size"), acc=("acc", "mean"))
)

flip_wide = pd.concat(
    {"acc": flip_perf["acc"].unstack("streak_bucket"),
     "n":   flip_perf["n"].unstack("streak_bucket")},
    axis=1
)

# order: +1 acc, -1 acc, +1 n, -1 n, ... +3 acc, -3 acc, +3 n, -3 n, 3+ acc, -3+ acc, 3+ n, -3+ n
ordered_cols = []
for k in [1, 2, 3, "3+"]:
    pb = (K + 1) if k == "3+" else k
    nb = -(K + 1) if k == "3+" else -k
    ordered_cols += [("acc", pb), ("acc", nb), ("n", pb), ("n", nb)]

flip_wide = flip_wide.reindex(columns=pd.MultiIndex.from_tuples(ordered_cols))

# relabel buckets
rename_cols = []
for metric, b in flip_wide.columns:
    if b == (K + 1): lab = "3+"
    elif b == -(K + 1): lab = "-3+"
    else: lab = str(b)
    rename_cols.append((metric, lab))
flip_wide.columns = pd.MultiIndex.from_tuples(rename_cols)

flip_wide.round(2)

acc         n     acc         n     \
                                           1    -1   1 -1    2    -2   2 -2   
model         train_years feature_set                                         
random_forest 4           daily         0.36  0.00  11  9  0.7  0.38  10  8   
                          daily+minute  0.55  0.00  11  9  0.5  0.38  10  8   
                          minute        0.91  0.00  11  9  0.8  0.12  10  8   
              6           daily         0.27  0.00  11  9  0.7  0.38  10  8   
                          daily+minute  0.55  0.00  11  9  0.6  0.25  10  8   
                          minute        1.00  0.00  11  9  0.9  0.00  10  8   
xgboost       4           daily         0.55  0.00  11  9  0.8  0.38  10  8   
                          daily+minute  0.64  0.00  11  9  0.4  0.25  10  8   
                          minute        0.73  0.11  11  9  0.6  0.12  10  8   
              6           daily         0.45  0.00  11  9  0.8  0.62  10  8   
                          daily+minute  0.64  0.00  11  9  0.6  0.50  10  8   
                          minute        1.00  0.22  11  9  0.5  0.38  10  8   

                                        acc         n      acc          n      
                                          3    -3   3 -3    3+   -3+   3+ -3+  
model         train_years feature_set                                          
random_forest 4           daily         1.0  1.00  10  7  0.97  0.96  117  56  
                          daily+minute  0.9  0.71  10  7  0.95  0.93  117  56  
                          minute        0.8  0.14  10  7  0.95  0.12  117  56  
              6           daily         1.0  0.71  10  7  0.98  0.96  117  56  
                          daily+minute  0.9  0.57  10  7  0.98  0.91  117  56  
                          minute        1.0  0.00  10  7  0.97  0.11  117  56  
xgboost       4           daily         1.0  0.71  10  7  0.97  0.91  117  56  
                          daily+minute  0.9  0.29  10  7  0.96  0.88  117  56  
                          minute        0.8  0.14  10  7  0.82  0.27  117  56  
              6           daily         1.0  0.57  10  7  0.95  0.95  117  56  
                          daily+minute  0.9  0.43  10  7  0.96  0.77  117  56  
                          minute        0.9  0.00  10  7  0.80  0.25  117  56